In the previous notebook I explored the Rossmann sales data.
In this notebook I focus on feature engineering, model training and evaluation for the sales forecasting task.

In [24]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as pt 
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import BaseEstimator, TransformerMixin

In [4]:
df_sales = pd.read_csv('../data/train.csv', low_memory=False)
df_store = pd.read_csv('../data/store.csv', low_memory=False)

In [6]:
def merge_store(df_store : pd.DataFrame, df_sales : pd.DataFrame) -> pd.DataFrame:
    df_store_copy = df_store.copy()
    df_sales_copy = df_sales.copy()
    df_sales_copy['Date'] = pd.to_datetime(df_sales_copy['Date'])
    df_full = pd.merge(df_store_copy, df_sales_copy, on='Store', how='left')
    return df_full 

df_full = merge_store(df_store, df_sales)

In [7]:
df_full.info()
df_full.head()

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   StoreType                  1017209 non-null  str           
 2   Assortment                 1017209 non-null  str           
 3   CompetitionDistance        1014567 non-null  float64       
 4   CompetitionOpenSinceMonth  693861 non-null   float64       
 5   CompetitionOpenSinceYear   693861 non-null   float64       
 6   Promo2                     1017209 non-null  int64         
 7   Promo2SinceWeek            509178 non-null   float64       
 8   Promo2SinceYear            509178 non-null   float64       
 9   PromoInterval              509178 non-null   str           
 10  DayOfWeek                  1017209 non-null  int64         
 11  Date                       1017209 non-null  dat

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,5,2015-07-31,5263,555,1,1,0,1
1,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,4,2015-07-30,5020,546,1,1,0,1
2,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,3,2015-07-29,4782,523,1,1,0,1
3,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2,2015-07-28,5011,560,1,1,0,1
4,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,1,2015-07-27,6102,612,1,1,0,1


In [14]:
df_full = df_full.sort_values('Date')

y = df_full['Sales'].copy() 
X = df_full.drop(columns=['Sales'])

In [17]:
X['Date'].head(3000)

1017208   2013-01-01
679363    2013-01-01
155193    2013-01-01
632403    2013-01-01
361623    2013-01-01
             ...    
422711    2013-01-03
65797     2013-01-03
147081    2013-01-03
129161    2013-01-03
907662    2013-01-03
Name: Date, Length: 3000, dtype: datetime64[us]

In [18]:
X.info()

<class 'pandas.DataFrame'>
Index: 1017209 entries, 1017208 to 0
Data columns (total 17 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int64         
 1   StoreType                  1017209 non-null  str           
 2   Assortment                 1017209 non-null  str           
 3   CompetitionDistance        1014567 non-null  float64       
 4   CompetitionOpenSinceMonth  693861 non-null   float64       
 5   CompetitionOpenSinceYear   693861 non-null   float64       
 6   Promo2                     1017209 non-null  int64         
 7   Promo2SinceWeek            509178 non-null   float64       
 8   Promo2SinceYear            509178 non-null   float64       
 9   PromoInterval              509178 non-null   str           
 10  DayOfWeek                  1017209 non-null  int64         
 11  Date                       1017209 non-null  datetime

In [20]:
print(X['Date'].min(), X['Date'].max())

2013-01-01 00:00:00 2015-07-31 00:00:00


In [21]:
split_date = '2015-04-01'

mask_train = df_full['Date'] < split_date
mask_test = df_full['Date'] >= split_date

X_train = X[mask_train]
X_test = X[mask_test]
y_train = y[mask_train]
y_test = y[mask_test]

In [22]:
print(f'Train dataset shape: {X_train.shape}')
print(f'Test dataset shape: {X_test.shape}')

Train dataset shape: (881179, 17)
Test dataset shape: (136030, 17)


In [23]:
print(y_train.shape, y_test.shape)

(881179,) (136030,)


In [25]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 881179 entries, 1017208 to 17836
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      881179 non-null  int64         
 1   StoreType                  881179 non-null  str           
 2   Assortment                 881179 non-null  str           
 3   CompetitionDistance        878903 non-null  float64       
 4   CompetitionOpenSinceMonth  601019 non-null  float64       
 5   CompetitionOpenSinceYear   601019 non-null  float64       
 6   Promo2                     881179 non-null  int64         
 7   Promo2SinceWeek            439516 non-null  float64       
 8   Promo2SinceYear            439516 non-null  float64       
 9   PromoInterval              439516 non-null  str           
 10  DayOfWeek                  881179 non-null  int64         
 11  Date                       881179 non-null  datetime64[us]
 12 

In [26]:
X_train['StateHoliday'].value_counts()

StateHoliday
0    856470
a     16149
b      4460
c      4100
Name: count, dtype: int64

In [27]:
X_train['DayOfWeek'].value_counts()

DayOfWeek
2    126709
4    125775
5    125775
6    125775
7    125775
1    125775
3    125595
Name: count, dtype: int64

In [28]:
class FeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self 
    
    def _has_competition(self, X):
        has_comp = (~X["CompetitionOpenSinceYear"].isna()) & (~X["CompetitionOpenSinceMonth"].isna())
        X["HasCompetition"] = has_comp.astype(int)
        
    
    def _calculate_months_since_competition(self, X):
        X["CompetitionOpenSinceYear_filled"] = X["CompetitionOpenSinceYear"].fillna(X["Date"].dt.year).astype(int)
        X["CompetitionOpenSinceMonth_filled"] = X["CompetitionOpenSinceMonth"].fillna(X["Date"].dt.month).astype(int)
        date_year = X["Date"].dt.year
        date_month = X["Date"].dt.month
        start_year = X["CompetitionOpenSinceYear_filled"]
        start_month = X["CompetitionOpenSinceMonth_filled"]
        X["MonthsSinceCompetition"] = (date_year - start_year) * 12 + (date_month - start_month)
        X.loc[X["HasCompetition"] == 0, "MonthsSinceCompetition"] = 0
        X["MonthsSinceCompetition"] = X["MonthsSinceCompetition"].clip(lower=0)
        
    
    def _calculate_months_since_promo2(self, X):
        has_promo2 = (
        (X["Promo2"] == 1)
        & (~X["Promo2SinceYear"].isna())
        & (~X["Promo2SinceWeek"].isna()))

        X["Promo2SinceYear_filled"] = X["Promo2SinceYear"].fillna(X["Date"].dt.isocalendar().year).astype(int)
        X["Promo2SinceWeek_filled"] = X["Promo2SinceWeek"].fillna(X["Date"].dt.isocalendar().week).astype(int)
        promo2_start_str = (
            X["Promo2SinceYear_filled"].astype(str)
            + X["Promo2SinceWeek_filled"].astype(str).str.zfill(2)
            + "1")
        X["Promo2StartDate"] = pd.to_datetime(
            promo2_start_str,
            format="%G%V%u",
            errors="coerce"
        )
        start_year = X["Promo2StartDate"].dt.year
        start_month = X["Promo2StartDate"].dt.month
        date_year = X["Date"].dt.year
        date_month = X["Date"].dt.month

        X["months_since_promo2"] = (date_year - start_year) * 12 + (
            date_month - start_month
        )
        X.loc[~has_promo2, "months_since_promo2"] = 0
        X["months_since_promo2"] = X["months_since_promo2"].fillna(0).clip(lower=0)
        

    
    def transform(self, X):
        X_copy = X.copy()
        X_copy['Year'] = X_copy['Date'].dt.year
        X_copy['Month'] = X_copy['Date'].dt.month
        X_copy['WeekOfYear'] = X_copy['Date'].dt.isocalendar().week.astype(int)
        X_copy['IsWeekend'] = np.where((X_copy['DayOfWeek'] == 6) | (X_copy['DayOfWeek'] == 7), 1, 0)
        self._has_competition(X_copy)
        self._calculate_months_since_competition(X_copy)
        self._calculate_months_since_promo2(X_copy)
        X_copy['Promo_SchoolHoliday_interaction'] = X_copy['Promo'] * X_copy['SchoolHoliday']
        X_copy['Promo_DayOfWeek_interaction'] = X_copy['Promo'] * X_copy['DayOfWeek']
        X_copy = X_copy.drop(columns=[
            "Date",
            "CompetitionOpenSinceMonth",
            "CompetitionOpenSinceYear",
            "Promo2SinceWeek",
            "Promo2SinceYear",
            "CompetitionOpenSinceYear_filled",
            "CompetitionOpenSinceMonth_filled",
            "Promo2SinceYear_filled",
            "Promo2SinceWeek_filled",
            "Promo2StartDate"])
        return X_copy

In [29]:
ft = FeatureTransformer()
X_val = ft.transform(X_train)
X_val.info()

<class 'pandas.DataFrame'>
Index: 881179 entries, 1017208 to 17836
Data columns (total 21 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Store                            881179 non-null  int64  
 1   StoreType                        881179 non-null  str    
 2   Assortment                       881179 non-null  str    
 3   CompetitionDistance              878903 non-null  float64
 4   Promo2                           881179 non-null  int64  
 5   PromoInterval                    439516 non-null  str    
 6   DayOfWeek                        881179 non-null  int64  
 7   Customers                        881179 non-null  int64  
 8   Open                             881179 non-null  int64  
 9   Promo                            881179 non-null  int64  
 10  StateHoliday                     881179 non-null  str    
 11  SchoolHoliday                    881179 non-null  int64  
 12  Year         

In [30]:
X_val.head()

,Store,StoreType,Assortment,CompetitionDistance,Promo2,PromoInterval,DayOfWeek,Customers,Open,Promo,...,SchoolHoliday,Year,Month,WeekOfYear,IsWeekend,HasCompetition,MonthsSinceCompetition,months_since_promo2,Promo_SchoolHoliday_interaction,Promo_DayOfWeek_interaction
1017208,1115,d,c,5350.0,1,"Mar,Jun,Sept,Dec",2,0,0,0,...,1,2013,1,1,0,0,0,8,0,0
679363,746,d,c,4330.0,1,"Mar,Jun,Sept,Dec",2,0,0,0,...,1,2013,1,1,0,1,23,17,0,0
155193,171,a,a,2640.0,0,NaN,2,0,0,0,...,1,2013,1,1,0,0,0,0,0,0
632403,694,a,c,460.0,1,"Jan,Apr,Jul,Oct",2,0,0,0,...,1,2013,1,1,0,1,2,0,0,0
361623,396,a,c,23130.0,0,NaN,2,0,0,0,...,1,2013,1,1,0,0,0,0,0,0


In [37]:
cols = ['Year', 'Month', 'WeekOfYear', 'IsWeekend', 'HasCompetition', 'MonthsSinceCompetition', 'months_since_promo2', 'Promo_SchoolHoliday_interaction', 'Promo_DayOfWeek_interaction']

for col in cols:
    print(f'\n{X_val[col].value_counts()}')


Year
2013    406974
2014    373855
2015    100350
Name: count, dtype: int64

Month
3     103695
1     103694
2      93660
5      69130
4      66900
6      66900
7      63550
8      63550
10     63550
12     63550
9      61500
11     61500
Name: count, dtype: int64

WeekOfYear
2     23415
3     23415
4     23415
5     23415
6     23415
7     23415
8     23415
9     23415
10    23415
11    23415
12    23415
13    23415
1     21759
14    17840
15    15610
16    15610
17    15610
18    15610
19    15610
20    15610
21    15610
22    15610
23    15610
24    15610
25    15610
26    15610
27    14530
28    14350
29    14350
30    14350
31    14350
32    14350
33    14350
34    14350
35    14350
36    14350
37    14350
38    14350
39    14350
40    14350
41    14350
42    14350
43    14350
44    14350
45    14350
46    14350
47    14350
48    14350
49    14350
50    14350
51    14350
52    14350
Name: count, dtype: int64

IsWeekend
0    629629
1    251550
Name: count, dtype: int64

HasCompeti